# 🌱 Visualizador de Etapas de Crecimiento — Simulador

Este notebook expone **una sola función** `cambiar_etapa()` que permite
saltar directamente a cualquier etapa del ciclo de vida de la planta y
visualizar su estado completo.

### Etapas disponibles

| Etapa | Tipo de planta |
|---|---|
| `germinacion` | General y Orquídea |
| `enraizamiento` | General y Orquídea |
| `plantula` | General y Orquídea |
| `crecimiento` | General y Orquídea |
| `floracion` | General |
| `fructificacion` | General |
| `vara_floral` | Orquídea |
| `botones_florales` | Orquídea |
| `crecimiento_botones` | Orquídea |
| `apertura_petalos` | Orquídea |

> **Requisito:** el backend FastAPI debe estar corriendo en `http://localhost:8000`

## 1. Configuración inicial

Ajusta las credenciales y el `PLANT_ID` de la planta que quieres explorar.

In [ ]:
import requests
import json

BASE = "http://127.0.0.1:8000"

# ── Ajusta con tus credenciales ──────────────────────────────────────────────
CORREO   = "tu_correo@ejemplo.com"
PASSWORD = "tu_password"
PLANT_ID = 1   # ID de la planta a explorar
# ─────────────────────────────────────────────────────────────────────────────

resp = requests.post(
    f"{BASE}/auth/login",
    json={"correo": CORREO, "password": PASSWORD},
)
datos = resp.json()

if resp.status_code == 200:
    USER_ID = datos["usuario"]["id"]
    print(f"✓ Sesión iniciada — user_id={USER_ID}")
else:
    print("✗ Error al iniciar sesión:", datos)

## 2. Función `cambiar_etapa()`

La función cambia la etapa de la planta y muestra su estado completo
(niveles de agua, luz, humedad, salud, alertas activas y descripción de la etapa).

In [ ]:
ETAPAS_ICONOS = {
    "germinacion":         "🌰",
    "enraizamiento":       "🌱",
    "plantula":            "🪴",
    "crecimiento":         "🌿",
    "floracion":           "🌸",
    "fructificacion":      "🍎",
    "vara_floral":         "🎋",
    "botones_florales":    "🌼",
    "crecimiento_botones": "🌺",
    "apertura_petalos":    "🌷",
}


def cambiar_etapa(etapa: str, plant_id: int = PLANT_ID) -> dict:
    """
    Cambia la etapa de crecimiento de la planta indicada y muestra
    su estado completo en pantalla.

    Parámetros
    ----------
    etapa    : nombre de la etapa (ver tabla en celda 1)
    plant_id : ID de la planta (por defecto usa PLANT_ID global)

    Retorna
    -------
    dict con el estado completo devuelto por el API
    """
    # 1. Cambiar etapa
    r_etapa = requests.put(
        f"{BASE}/plants/{plant_id}/stage",
        json={"etapa": etapa},
    )
    if r_etapa.status_code != 200:
        print(f"✗ No se pudo cambiar la etapa: {r_etapa.json()}")
        return r_etapa.json()

    # 2. Consultar estado actualizado
    r_estado = requests.get(f"{BASE}/plants/{plant_id}/status")
    if r_estado.status_code != 200:
        print(f"✗ No se pudo obtener el estado: {r_estado.json()}")
        return r_estado.json()

    estado = r_estado.json()
    planta = estado["planta"]
    icono  = ETAPAS_ICONOS.get(etapa, "🌿")

    # 3. Visualización
    linea = "═" * 58
    print(f"\n{linea}")
    print(f"  {icono}  Etapa: {etapa.upper()}")
    print(f"  {estado['etapa_descripcion']}")
    print(linea)
    print(f"  Planta          : {planta['plant_name']} (ID {planta['id_plant']})")
    print(f"  Tipo            : {planta['plant_type']}")
    print(f"  Acciones totales: {planta['total_care_actions']}")
    print()
    print(f"  💧 Agua         : {planta['water_level']:.1f} %")
    print(f"  ☀️  Luz          : {planta['light_level']:.1f} %")
    print(f"  💦 Humedad      : {planta['humidity_level']:.1f} %")
    print(f"  🌬️  Ventilación  : {planta['ventilation_level']:.1f} %")
    print(f"  ❤️  Salud        : {planta['health']:.1f} % — {estado['salud_etiqueta']}")
    print(f"  🪨  Sustrato     : {planta['substrate_name']}")

    alertas = estado.get("alertas", [])
    if alertas:
        print()
        print("  ⚠️  Alertas activas:")
        for alerta in alertas:
            print(f"     • {alerta}")
    else:
        print()
        print("  ✅ Sin alertas — condiciones óptimas")

    print(linea)
    return estado


print("Función cambiar_etapa() lista ✓")

## 3. Explorar etapas

Ejecuta la celda que corresponda a la etapa que quieres visualizar,
o modifica el argumento libremente.

In [ ]:
cambiar_etapa("germinacion")

In [ ]:
cambiar_etapa("enraizamiento")

In [ ]:
cambiar_etapa("plantula")

In [ ]:
cambiar_etapa("crecimiento")

In [ ]:
# Orquídea
cambiar_etapa("vara_floral")

In [ ]:
# Orquídea
cambiar_etapa("botones_florales")

In [ ]:
# Orquídea
cambiar_etapa("crecimiento_botones")

In [ ]:
# Orquídea
cambiar_etapa("apertura_petalos")

## 4. Recorrido completo automático

Ejecuta esta celda para ver **todas las etapas** del ciclo de la orquídea en secuencia.

In [ ]:
ETAPAS_ORQUIDEA = [
    "germinacion",
    "enraizamiento",
    "plantula",
    "crecimiento",
    "vara_floral",
    "botones_florales",
    "crecimiento_botones",
    "apertura_petalos",
]

for etapa in ETAPAS_ORQUIDEA:
    cambiar_etapa(etapa)